# STINTSY Major Course Output

Predict successful heart transplantation using the ORCHID dataset.

# Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

%matplotlib inline

# Load Dataset

In [ ]:
na_values = ['', ' ', 'na ', 'na']

OPOreferrals = pd.read_csv('orchid-2.1.1/OPOReferrals.csv', na_values=na_values, low_memory=False)
CBCEvents = pd.read_csv('orchid-2.1.1/CBCEvents.csv', na_values=na_values, low_memory=False)
ChemistryEvents = pd.read_csv('orchid-2.1.1/ChemistryEvents.csv', na_values=na_values, low_memory=False)

# Data Cleaning

In [ ]:
OPOreferrals['target'] = (OPOreferrals['outcome_heart'] == 'Transplanted').astype(int)
df_cohort = OPOreferrals[OPOreferrals['procured'] == True].copy()

print('Total donors:', len(df_cohort))
print('Successful transplants:', df_cohort['target'].sum())

# Feature Engineering

In [ ]:
cbc_features = CBCEvents.sort_values('time_event').groupby(['patient_id', 'cbc_name'])['value'].last().unstack()

chem_features = ChemistryEvents.sort_values('time_event').groupby(['patient_id', 'chem_name'])['value'].last().unstack()

df_final = df_cohort.merge(cbc_features, on='patient_id', how='left')
df_final = df_final.merge(chem_features, on='patient_id', how='left')

df_final.head()

# Data Preparation

In [ ]:
features = ['age', 'weight_kg', 'WBC', 'Hgb', 'Creatinine']

X = df_final[features].fillna(0)
y = df_final['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model 1: Logistic Regression

In [ ]:
model_lr = LogisticRegression(max_iter=1000)
model_lr.fit(X_train, y_train)

pred_lr = model_lr.predict(X_test)
print(classification_report(y_test, pred_lr))

# Model 2: Random Forest

In [ ]:
model_rf = RandomForestClassifier(n_estimators=100, random_state=42)
model_rf.fit(X_train, y_train)

pred_rf = model_rf.predict(X_test)
print(classification_report(y_test, pred_rf))

# Model Evaluation

In [ ]:
print('Logistic Regression:')
print(classification_report(y_test, pred_lr))

print('\nRandom Forest:')
print(classification_report(y_test, pred_rf))